_ExplainDB — Copyright (C) 2026 Prof. Dr. Jens Dittrich, Saarland University._  
_Licensed under the GNU Affero General Public License v3 (AGPL-3.0). See the `LICENSE` file._

# MVCC Playground

Build schedules of transactions on the MVCC store of this repository and run them operation by operation.

- **Scenario:** choose a classic schedule, or **empty** to build your own. Once you change a scenario, the dropdown
  shows **own schedule**; choose the scenario again to start over.
- **Build:** choose a transaction and an operation (and a key and values where they apply), then click **add**.
  Only operations that are valid at that point are offered: a new transaction first begins, and a transaction whose
  commit or abort is in the schedule gets no further operations. **remove last** takes back the last operation.
- **Mode:** how the store decides which transactions to abort. **serializable - method 1 (re-run reads,
  checksums)** and **serializable - method 2 (write sets)** are two ways to implement the isolation level
  serializable: at the commit of a transaction that wrote something, the store checks whether a transaction that
  committed meanwhile changed something this transaction read (method 1 runs every read again and compares the
  checksums of the results; method 2 checks the reads against the objects written by those transactions); a
  transaction that wrote nothing is not checked. **snapshot isolation - first updater wins** does not check reads;
  instead, a transaction is aborted at once when it writes an object that another transaction committed after it
  began. **no validation** aborts a transaction only for the rule all modes share: a write fails while another
  transaction holds an uncommitted version of the object. In every mode, transactions read the snapshot of their
  begin plus their own writes. With method 1, the transactions table shows the checksum of each read when it ran and
  the checksum the same read gives now; a commit now fails if they differ (red), unless the transaction wrote nothing.
- **Run:** click an operation in the schedule to run everything up to and including it, or use **back** and
  **forward**. Operations that ran are blue, planned ones faint and in italics, and the one that ran last is framed
  (and yellow if it ran normally and is not a commit). A commit that succeeded is green; red marks where the store
  aborted a transaction, which then skips its remaining operations (grey); orange marks an error that stops the run.
  Once every transaction has ended, violet marks the read that returns something else in the order of the
  serialization points. The column **why** next to the transactions says what that read would return there, and why
  the store aborted a transaction.
- Below the schedule: the versions in the store, each transaction's status, its snapshot plus its own writes, what
  it read and wrote, and the verdict: whether the result is serializable and, unless it is in the order of the
  serialization points, one line per serial order that was replayed.

In [1]:
import sys, pathlib

# Put the repo root (the directory containing `system/`) on sys.path so
# `from system...` works whether this notebook is opened from its own
# folder or from the repo root.
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "system").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

In [2]:
from system.playground.mvcc_widgets import MvccPlayground

MvccPlayground().widget

## Things to try

- **lost update**: run it in each mode. Method 1 or 2 aborts T2 at its commit, snapshot isolation already at its
  write, and without validation both commit. Why is that result not serializable?
- **write skew**, **write skew via phantom (insert and delete)**: method 1 or 2 aborts T2; snapshot isolation commits
  both, as they write different objects. Snapshot isolation prevents lost updates, but not write skew.
- **read-only anomaly (Fekete et al.)**: under snapshot isolation all three transactions commit, and the result is
  not serializable; even T3, which only reads, sees the deposit but not the withdrawal, a state no serial order
  produces.
- **phantom insert**, **concurrent delete**: under snapshot isolation and without validation the result is
  serializable, but in another order than the order of the serialization points (snapshot isolation does not promise
  that order). Is the abort by method 1 or 2 necessary?
- **blind write**: method 1 or 2 commits both transactions, snapshot isolation aborts T1. Snapshot isolation does
  not simply abort less than the serializable methods.